# 04 — SPURS stability scores

Default: load the cached **raw ddG** table `scores/5lpg_spurs.csv`. Optional live inference uses `get_SPURS_from_hub` + `parse_pdb_resolved_only` (not the evedesign SPURS class). Do not `pip install spurs` from PyPI.


In [ ]:
from pathlib import Path

from dualde.mapping import load_structural_mapping_json, read_fasta, select_mapping_entry
from dualde.paths import example_dir
from dualde.scores_spurs import add_neg_spurs, load_spurs_table, run_spurs_scan

EXAMPLE = example_dir("NUDT15_HUMAN")
OUT = EXAMPLE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

RUN_SPURS = False  # flip to True to ignore the cache
CACHE = EXAMPLE / "scores" / "5lpg_spurs.csv"

if RUN_SPURS:
    header, uniprot_seq = read_fasta(EXAMPLE / "NUDT15_HUMAN.fasta")
    entry = select_mapping_entry(
        load_structural_mapping_json(EXAMPLE / "mapping" / "5lpg_A.json"),
        pdb_id="5lpg",
        chain="A",
        uniprot_id="NUD15_HUMAN",
    )
    spurs = run_spurs_scan(
        str(EXAMPLE / "pdb" / "5lpg_cleaned.pdb"),
        entry,
        chain="A",
        uniprot_sequence=uniprot_seq,
        seqres_search_dirs=[str(EXAMPLE / "pdb")],
    )
    spurs.to_csv(OUT / "5lpg_spurs.csv", index=False)
else:
    spurs = load_spurs_table(CACHE)
    print("Loaded cache", CACHE)

spurs = add_neg_spurs(spurs)
print(spurs.shape)
print(spurs.head())
spurs.to_csv(OUT / "5lpg_spurs.csv", index=False)
